In [2]:
from model import VAE_Encoder_MNIST,Ganerator_MNIST
import torch
from torchvision import datasets, transforms
from torchvision.utils import save_image
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

In [1]:
import com_utils
import time

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(
    root="D:/Python/SemCom/saved_data",
    train=True,
    download=False,
    transform=transform
)

# dataloader
dl = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=False)

fixed_x, _ = next(iter(dl))
fixed_x = fixed_x[:64].to(device)

In [6]:
z_dim = 64
E = VAE_Encoder_MNIST(latent_dim=z_dim).to(device)
E.load_state_dict(torch.load("./save_model/E_invert_to_G_C_only.pth"))
E.eval()
for p in E.parameters():
    p.requires_grad = False

G = Ganerator_MNIST(z_dim=z_dim).to(device)
G.load_state_dict(torch.load("./save_model/G_wgangp_C.pth"))
G.eval()
for p in G.parameters():
    p.requires_grad = False

In [7]:
with torch.no_grad():
    mu, logvar = E(fixed_x)
    z = mu

# 存 latent
z_np = z.detach().cpu().numpy()

In [9]:
# quantization tensor
# [-128,127]
bits = 5
qmax = 2**(bits-1) - 1
qmin = -2**(bits-1)

scale = z.abs().max() / qmax

z_q = torch.round(z / scale).clamp(qmin, qmax).to(torch.int8)

z_q_b = com_utils.array_to_bits(z_q)
print(z_q_b)

# recover
z_rec = z_q.float() * scale

[0 1 0 ... 1 0 0]


In [ ]:

seq_image = 1
seq_image_max = 100
base_name="./photos3/IMG"
file_path = f"{base_name}_{seq_image:05d}.jpg"
with open(file_path, "rb") as f:
    A = np.frombuffer(f.read(), dtype=np.uint8)
# np.set_printoptions(threshold=np.inf)
# print(A)
t_total0 = time.perf_counter()
C = com_utils.array_to_bits(A)
t_total = time.perf_counter() - t_total0
print(f"Total time : {t_total:.6f} s")
# np.set_printoptions(threshold=np.inf)
# print(C)
# print("file:", file_path)
# print("A shape:", A.shape)   # byte 數
# print("C shape:", C.shape)   # bit 數 = A.shape[0] * 8
# print("next seq_image:", seq_image)
